# Datavidia Task-2 v3 — Inference & Submission Notebook

**Model:** Patch-Chamfer DINOv2-S/14 @336px + Hurricane anchor

**Features:** 1152-dim = `pmean_pre` (384) + `pmean_post` (384) + `topchg_delta` (384)

**Ensemble (weighted probability):** 0.45·MLP + 0.35·LR + 0.20·kNN, lalu override Hurricane dari anchor

**CV Macro-F1:** 0.6586 ± 0.0251 (5-fold, seed 42)

**Rule compliance:** notebook ini TIDAK melatih ulang apa pun. Ia memuat artefak model yang sudah dilatih (`.onnx` + `.joblib`) dan hanya menghasilkan `submission.csv`.

**Artefak yang dibutuhkan (harus ada di folder kerja):**
```
cache/features/patch_chamfer_v3_dinov2_vits14_336.npz
cache/features/clusters_v3.npz
models/mlp_v3.onnx
models/lr_v3.onnx
models/knn_v3.joblib
models/scaler_v3.joblib
models/label_encoder_v3.joblib
data/raw/sample_submission.csv
```

In [ ]:
import json
from collections import Counter
from pathlib import Path

import joblib
import numpy as np
import onnxruntime as ort
import pandas as pd

SEED = 42
np.random.seed(SEED)

ROOT = Path.cwd()
CACHE_DIR = ROOT / "cache" / "features"
MODELS_DIR = ROOT / "models"
SUB_DIR = ROOT / "submissions"
SUB_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["Earthquake", "Explosion", "Fire", "Hurricane", "Tsunami", "Volcano"]

print(f"ROOT      : {ROOT}")
print(f"CACHE_DIR : {CACHE_DIR}")
print(f"MODELS_DIR: {MODELS_DIR}")

In [ ]:
# 1) Load patch-chamfer features -> X (N, 1152)
pc_path = CACHE_DIR / "patch_chamfer_v3_dinov2_vits14_336.npz"
pc = np.load(pc_path)
print("NPZ arrays:", list(pc.keys()))

X = np.concatenate(
    [pc["pmean_pre"], pc["pmean_post"], pc["topchg_delta"]], axis=1
).astype(np.float32)
print(f"Feature shape: {X.shape}")

In [ ]:
# 2) Load trained models (no retraining)
mlp_sess = ort.InferenceSession(str(MODELS_DIR / "mlp_v3.onnx"), providers=["CPUExecutionProvider"])
lr_sess = ort.InferenceSession(str(MODELS_DIR / "lr_v3.onnx"), providers=["CPUExecutionProvider"])

knn = joblib.load(MODELS_DIR / "knn_v3.joblib")
scaler = joblib.load(MODELS_DIR / "scaler_v3.joblib")
label_encoder = joblib.load(MODELS_DIR / "label_encoder_v3.joblib")

print("MLP ONNX outputs:", [(o.name, o.shape) for o in mlp_sess.get_outputs()])
print("LR  ONNX outputs:", [(o.name, o.shape) for o in lr_sess.get_outputs()])
print("kNN:", knn.__class__.__name__, "| metric =", knn.metric)
print("Classes:", list(label_encoder.classes_))

In [ ]:
# 3) Hurricane anchor mask (precomputed in clusters_v3.npz)
clusters = np.load(CACHE_DIR / "clusters_v3.npz")
hurricane_mask = clusters["hurricane_mask"]
print(f"Hurricane anchor samples: {int(hurricane_mask.sum())}")

In [ ]:
# 4) Weighted-probability ensemble (identical to training-time predict_all)
X_scaled = scaler.transform(X).astype(np.float32)

# ONNX: output[0] = label, output[1] = probabilities (N, 6)
mlp_proba = mlp_sess.run(None, {"input": X_scaled})[1].astype(np.float32)
lr_proba = lr_sess.run(None, {"input": X_scaled})[1].astype(np.float32)

# kNN probabilities (cosine metric on unscaled X)
knn_proba = knn.predict_proba(X).astype(np.float32)

# kNN was fitted on LabelEncoder integer codes (0..5) -> already aligned with label_encoder order
knn_classes = np.asarray(knn.classes_)
if knn_classes.dtype.kind in "iu":
    order = np.argsort(knn_classes.astype(int))
else:
    order = np.array([list(knn_classes).index(c) for c in label_encoder.classes_])
if not np.array_equal(order, np.arange(len(order))):
    knn_proba = knn_proba[:, order]
    print(f"Reordered kNN columns by {order.tolist()}")
else:
    print("kNN columns already aligned with label_encoder (identity order)")

ensemble_proba = 0.45 * mlp_proba + 0.35 * lr_proba + 0.20 * knn_proba
predictions = label_encoder.classes_[np.argmax(ensemble_proba, axis=1)]

print("Ensemble proba shape:", ensemble_proba.shape)
print("Row sums (should be ~1):", np.round(ensemble_proba.sum(axis=1)[:5], 4))


In [ ]:
# 5) Apply Hurricane anchor override
predictions = np.array(predictions, dtype=object)
predictions[hurricane_mask] = "Hurricane"

unique, counts = np.unique(predictions, return_counts=True)
print("Final prediction distribution:")
for u, c in zip(unique, counts):
    print(f"  {u:<12} {c}")

In [ ]:
# 6) Build submission aligned to sample_submission.csv Id order
sample_sub = pd.read_csv(ROOT / "data" / "raw" / "sample_submission.csv")
assert len(predictions) == len(sample_sub), f"Mismatch: {len(predictions)} vs {len(sample_sub)}"

submission = pd.DataFrame({"Id": sample_sub["Id"], "label": predictions})
output_path = SUB_DIR / "submission_v3.csv"
submission.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows : {len(submission)}")
submission.head(10)

In [ ]:
# 7) Validate submission
print("Label distribution:")
print(submission["label"].value_counts())
print()
print("All 6 classes present :", set(submission["label"].unique()) == set(CLASS_NAMES))
print("No nulls             :", bool(submission.notna().all().all()))
print("Id unique            :", submission["Id"].nunique() == len(submission))
print("Columns              :", list(submission.columns))